 ### **Fase 01: Infraestructura en AWS y Pipeline de Datos**
**Objetivo:** Construir un sistema automatizado que extraiga datos de la API de TMDB y los almacene de forma estructurada en una base de datos en la nube.

**Extración masiva**

In [ ]:
BUCKET = "tmbd-peliculas-nuevas "
LIST_SIZE = 10_000

s3 = boto3.client("s3", region_name = "eu-north-1")

In [ ]:
def get_movie(movie_id):
    api_key = "8a1ba0b66e4fae48f3d50b6e0437865b"
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={api_key}"

    response = requests.get(url)

    if response.status_code != 200:
        return None

    data = response.json()
    
    return data

In [ ]:
def upload_movie_list_to_s3(movie_list, start_id, end_id):
    try:
        s3.put_object(
            Bucket=BUCKET,
            Key=f"movies_{start_id}-{end_id}.json",
            Body=json.dumps(movie_list)
        )
        print(f"Subido bloque {start_id} - {end_id}")


    except:
        s3.put_object(
            Bucket=BUCKET,
            Key="ultima-extraccion.txt", 
            Body=f"{end_id - len(movie_list)}")

In [ ]:
def extract_movies(start_id=757353, end_id=866100):
    movie_list = []
    list_start = start_id


    for movie_id in range(start_id, end_id + 1):
        data = get_movie(movie_id)


        if data:
            movie_list.append(data)


        if len(movie_list) >= LIST_SIZE:
            upload_movie_list_to_s3(movie_list, list_start, movie_id)
            movie_list = []
            list_start = movie_id + 1


        if movie_id % 5000 == 0:
            s3.put_object(
                Bucket=BUCKET,
                Key="checkpoint.txt",
                Body=str(movie_id)
            )


    if movie_list:
        upload_movie_list_to_s3(movie_list, list_start, end_id)

**Extracción/limpieza diaria (LAMBDA)**

In [ ]:
def data_cleanup(data):

    data = data.fillna("None")
    data = data.replace("", "None")
    data = data.replace("Sin dato", "None")

    data["spoken_languages"] = data["spoken_languages"].apply(
        lambda x: [i["name"] for i in x] if isinstance(x, list) else x
    )
    data["genres"] = data["genres"].apply(
        lambda x: [i["name"] for i in x] if isinstance(x, list) else x
    )
    data["production_countries"] = data["production_countries"].apply(
        lambda x: [i["name"] for i in x] if isinstance(x, list) else x
    )
    data["production_companies"] = data["production_companies"].apply(
        lambda x: [i["name"] for i in x] if isinstance(x, list) else x
    )
    data["belongs_to_collection"] = data["belongs_to_collection"].apply(
        lambda x: x["name"] if isinstance(x, dict) else x
    )
    data["release_date"] = pd.to_datetime(
        data["release_date"], errors="coerce"
    )

    return data
    

def extraer_y_transformar(bucket, key):
    
    s3 = boto3.client("s3")
    response = s3.get_object(Bucket=bucket, Key=key)
    data = json.loads(response["Body"].read().decode("utf-8"))

    movies = []
    genres = {}
    movie_genres = []

    for movie in data:
        movie_id = movie["id"]

        movies.append({
            "movie_id": movie_id,
            "title": movie.get("title"),
            "original_title": movie.get("original_title"),
            "overview": movie.get("overview"),
            "release_date": movie.get("release_date"),
            "runtime": movie.get("runtime"),
            "budget": movie.get("budget"),
            "revenue": movie.get("revenue"),
            "popularity": movie.get("popularity"),
            "vote_average": movie.get("vote_average"),
            "vote_count": movie.get("vote_count"),
            "status": movie.get("status"),
            "tagline": movie.get("tagline"),
            "imdb_id": movie.get("imdb_id"),
            "original_language": movie.get("original_language"),
            "adult": movie.get("adult"),
            "video": movie.get("video")
        })

        
        for genre in movie.get("genres", []):
            genres[genre["id"]] = genre["name"]
            movie_genres.append({
                "movie_id": movie_id,
                "genre_id": genre["id"]
            })

    
    df_movies = pd.DataFrame(movies)
    df_genres = pd.DataFrame([{"genre_id": k, "name": v} for k, v in genres.items()])
    df_movie_genres = pd.DataFrame(movie_genres)

    
    df_movies["release_date"] = pd.to_datetime(df_movies["release_date"], errors="coerce")

    return df_movies, df_genres, df_movie_genres

def get_connection():
    return psycopg2.connect(
        username=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        database=os.getenv("DB_NAME")
    )

def cargar_datos(df: pd.DataFrame) -> None:
    
    columnas = list(df.columns)
    placeholders = sql.SQL(", ").join([sql.Placeholder()] * len(columnas))
    columnas_sql = sql.SQL(", ").join(map(sql.Identifier, columnas))

    insert_query = sql.SQL("""
        INSERT INTO {table} ({cols})
        VALUES ({vals})
    """).format(
        table=sql.Identifier(tabla),
        cols=columnas_sql,
        vals=placeholders
    )

    data = df.to_numpy().tolist()

    with get_connection() as conn:
        with conn.cursor() as cursor:
            cursor.executemany(insert_query, data)
            conn.commit()


def lambda_handler(event, context):

    if "Records" not in event:
        print("Evento sin 'Records'.")
        print("Evento recibido:", event)
        return {"status": "ERROR", "message": "Evento no contiene Records"}
    
    for record in event["Records"]:

        if "s3" not in record or "bucket" not in record["s3"] or "object" not in record["s3"]:
            print("Record inválido:", record)
            continue

        bucket = record["s3"]["bucket"]["name"]
        key = record["s3"]["object"]["key"]

        print(f"Procesando archivo: {key}")

        
        df_movies, df_genres, df_movie_genres = extraer_y_transformar(bucket, key)

        
        if not df_movies.empty:
            cargar_datos(df_movies, "movies")
            cargar_datos(df_genres, "genres")
            cargar_datos(df_movie_genres, "movie_genres")
            print(f"Carga finalizada para {key}")
        else:
            print(f"No se encontraron datos en {key}")


### **Fase 02: Modelado, API y Despliegue**
**Objetivo:** Desarrollar una API multifuncional que prediga el éxito de una película y permita hacer consultas complejas.


**Tareas Clave:**
1.  **Desarrollo del Modelo (Data Scientist):**
    - Utilizar los datos de PostgreSQL para entrenar un modelo de Machine Learning (ej. con Scikit-learn) que prediga si una película será un "éxito".


2.  **Desarrollo de la API (ML Engineer, Data Scientist):**
    - Crear una aplicación con **FastAPI** que incluya los siguientes endpoints:
      - `/predict`: Recibe los datos de una película y devuelve la predicción del modelo.
      - `/ask-text`: Recibe una pregunta en texto (ej. "¿Películas de mayor presupuesto de 2023?"), la convierte a SQL usando un modelo de **Hugging Face**, consulta la base de datos y devuelve una respuesta en texto.
      - `/ask-visual`: Recibe una pregunta orientada a visualización (ej. "Top 5 géneros por popularidad"), consulta la base de datos y devuelve un gráfico generado con **Matplotlib/Seaborn**.


3.  **Despliegue (ML Engineer):**
    - Desplegar la aplicación FastAPI completa en una instancia **AWS EC2** para que sea accesible públicamente.


**Entregables de esta fase:**
- Un modelo de clasificación entrenado.
- API funcional desplegada en EC2 con los tres endpoints implementados.
- Documentación de la API (generada por FastAPI).

**MODELO**

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, roc_auc_score
import pickle
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()

def get_connection():
    return psycopg2.connect(
        user=os.getenv("DB_USER"), 
        password=os.getenv("DB_PASSWORD"), 
        host=os.getenv("DB_HOST"), 
        port=os.getenv("DB_PORT"), 
        dbname=os.getenv("DB_NAME") 
    )


engine = get_connection()


In [ ]:
#cargar datos 

QUERY = """
SELECT
    m.movie_id,
    m.budget,
    m.revenue,
    m.runtime,
    m.popularity,
    m.vote_count,
    EXTRACT(YEAR FROM m.release_date) AS release_year,
    m.adult,
    m.video,
    m.title,
    m.overview,
    m.tagline,
    g.name AS genre_name
FROM movies m
LEFT JOIN movie_genres mg ON m.movie_id = mg.movie_id
LEFT JOIN genres g ON mg.genre_id = g.genre_id
WHERE m.budget > 0 AND m.revenue > 0;
"""
df_raw = pd.read_sql(QUERY, engine)


#target
df_raw['success'] = (df_raw['revenue'] > df_raw['budget'] * 2).astype(int)

#features
df_raw['adult'] = df_raw['adult'].astype(int)
df_raw['video'] = df_raw['video'].astype(int)
numeric_features = ['budget', 'runtime', 'popularity', 'vote_count', 'release_year', 'adult', 'video']

df_movies_unique = df_raw.drop_duplicates(subset='movie_id').set_index('movie_id')
X_num = df_movies_unique[numeric_features].fillna(df_movies_unique[numeric_features].median())

#one-hot
genre_ohe = df_raw[['movie_id', 'genre_name']].dropna()
genre_ohe['value'] = 1
genres_df = genre_ohe.pivot_table(
    index='movie_id',
    columns='genre_name',
    values='value',
    aggfunc='max',
    fill_value=0
)
genres_df = genres_df.reindex(df_movies_unique.index, fill_value=0)
genre_cols = genres_df.columns



#texto
df_movies_unique['text_combined'] = (
    df_movies_unique['title'].fillna("") + " " +
    df_movies_unique['overview'].fillna("") + " " +
    df_movies_unique['tagline'].fillna("")
)

tfidf = TfidfVectorizer(max_features=256, stop_words='english') 
X_tfidf = tfidf.fit_transform(df_movies_unique['text_combined']).toarray()


pca = PCA(n_components=64, random_state=42)  
X_text = pca.fit_transform(X_tfidf).astype('float32')



# Dataset final
X = np.hstack([X_num.values, genres_df.values, X_text])
y = df_movies_unique['success'].values



# train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)



# Escalado
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



# Modelo
gb = GradientBoostingClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
gb.fit(X_train_scaled, y_train)



#Comprobación
y_pred = gb.predict(X_test_scaled)
y_prob = gb.predict_proba(X_test_scaled)[:, 1]
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))




# Guardado para FastAPI
package = {
    'numeric_features': numeric_features,
    'genre_columns': list(genre_cols),
    'scaler': scaler,
    'tfidf_vectorizer': tfidf,
    'pca': pca,
    'gb_model': gb
}


with open('movie_model.pkl', 'wb') as f:
    pickle.dump(package, f)


print("Modelo para FastAPI guardado")

**FAST API**

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from io import BytesIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import requests
import psycopg2
from google import genai
import google.generativeai as genai
from google.generativeai.types import GenerationConfig


# Inicialización FastAPI

app = FastAPI(title="TMDB API")


# modelo y preprocesadores

try:
    with open("movie_model.pkl", "rb") as f:
        package = pickle.load(f)


    numeric_features = package['numeric_features']
    genre_columns = package['genre_columns']
    scaler = package['scaler']
    tfidf_vectorizer = package['tfidf_vectorizer']
    pca = package['pca']
    gb_model = package['gb_model']


except FileNotFoundError:
    print("movie_model.pkl no encontrado. El endpoint /predict no funcionará.")
    gb_model, scaler, tfidf_vectorizer, pca, numeric_features, genre_columns = [None]*6
    
# Conexión PostgreSQL 
def get_connection():
    return psycopg2.connect(
        user=os.getenv("DB_USER"), 
        password=os.getenv("DB_PASSWORD"), 
        host=os.getenv("DB_HOST"), 
        port=int(os.getenv("DB_PORT")), 
        dbname=os.getenv("DB_NAME")) 



# Modelos Pydantic

class MovieInput(BaseModel):
    title: str
    overview: str = ""
    tagline: str = ""
    budget: float
    runtime: float
    popularity: float
    vote_count: int
    release_year: int
    adult: bool
    video: bool
    genres: list[str] = []


class Question(BaseModel):
    text: str


# modelo Gemini 


GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY") # poner variable en .env
GEMINI_MODEL = "gemini-1.5-flash"          


genai.configure(api_key=GOOGLE_API_KEY)


def ask_gemini(prompt: str) -> str:
    """
    Función única para consultar a Gemini.
    Incluye configuración de temperatura baja para precisión
    y limpieza automática de formatos Markdown.
    """
    try:
        model = genai.GenerativeModel(GEMINI_MODEL)
        config = GenerationConfig(max_output_tokens=500, temperature=0.1)
        response = model.generate_content(prompt, generation_config=config)
        texto_limpio = response.text.replace("```sql", "").replace("```", "").strip()
        return texto_limpio
    except Exception as e:
        print(f"Error Gemini: {e}")
        return "ERROR_API"


# SQL Schema

sql_schema = """
TABLE genres (
    genre_id INTEGER,
    name TEXT
);

TABLE movie_genres (
    movie_id BIGINT,
    genre_id INTEGER
);

TABLE movies (
    movie_id BIGINT,
    title TEXT,
    original_title TEXT,
    overview TEXT,
    release_date DATE,
    runtime INT, 
    budget BIGINT,
    revenue BIGINT, 
    popularity DOUBLE PRECISION,
    vote_average DOUBLE PRECISION,
    vote_count INTEGER,
    status TEXT,
    tagline TEXT,
    imdb_id TEXT,
    original_language TEXT,
    adult BOOLEAN,
    video BOOLEAN
);
"""

# endpoint /predict
@app.get("/")
def bienvenido():
    return "Bienvenido a nuestra FastAPI de pelis!"
    
@app.post("/predict")
def predict_movie(movie: MovieInput):
    if gb_model is None:
        raise HTTPException(status_code=500, detail="Modelo no cargado")


    # features numéricas
    X_num = np.array([[movie.budget,
                       movie.runtime,
                       movie.popularity,
                       movie.vote_count,
                       movie.release_year,
                       int(movie.adult),
                       int(movie.video)]], dtype=float)


    # géneros one-hot
    X_genres = np.zeros((1, len(genre_columns)))
    movie_genres = [x.lower().strip() for x in movie.genres]
    for i, g in enumerate(genre_columns):
        if g.lower() in movie_genres:
            X_genres[0, i] = 1




    # texto
    text_combined = f"{movie.title} {movie.overview} {movie.tagline}"
    X_text_tfidf = tfidf_vectorizer.transform([text_combined]).toarray()
    X_text_pca = pca.transform(X_text_tfidf)


    # combina features
    X_final = np.hstack([X_num, X_genres, X_text_pca])
    X_final_scaled = scaler.transform(X_final)


    pred_class = int(gb_model.predict(X_final_scaled)[0])
    pred_prob = float(gb_model.predict_proba(X_final_scaled)[0, 1])


    return {
        "success_prediction": pred_class,
        "success_probability": pred_prob
    }




# endpoint /ask-text


@app.post("/ask-text")
def ask_text(question: Question) -> str:
    prompt = f"""
    Eres un bot experto en SQL. Tu base de datos tiene el siguiente esquema:
    {sql_schema}
    Pregunta del usuario:
    {question.text}
    Limita resultados a 50 filas.
    """
    # Gemini genera SQL
    respuesta_sql = ask_gemini(prompt_sql) 
    
    sql_lower = respuesta_sql.lower()
    forbidden = ["insert", "update", "delete", "drop", "alter", "truncate", "create", "grant", "revoke"]


    if not sql_lower.startswith("select") or any(w in sql_lower for w in forbidden):
        raise HTTPException(400, "Consulta SQL no permitida")


    with get_connection() as conn:
        df = pd.read_sql(respuesta_sql, conn)
    if df.empty:
        return "No hay resultados para esa consulta."


    prompt_respuesta = f"""
    Pregunta: {question.text}
    Datos encontrados: {df.head(10).to_dict(orient='records')}
    Tarea: Responde a la pregunta usando los datos. Eres un bot que responde de forma breve, clara y humana..
    """
    # Gemini genera la respuesta final
    respuesta_final = ask_gemini(prompt_respuesta)
    return respuesta_final


# endpoint /ask-visual


@app.post("/ask-visual")
def ask_visual(question: Question) -> StreamingResponse:
    prompt_graficos = f"""
    Eres un bot que genera SQL para visualización. Base de datos:
    {sql_schema}
    Pregunta: {question.text}
    """
    # Gemini genera SQL y tipo de gráfico
    respuesta = ask_gemini(prompt_graficos)
    
    lines = respuesta.splitlines()
    if len(lines) < 2:
        raise HTTPException(400, "Respuesta inválida del modelo")


    plot_type_raw = lines[0].strip().upper()
    plot_map = {"HISTOGRAMA": "hist", "SCATTERPLOT": "scatter", "BOXPLOT": "box"}
    if plot_type_raw not in plot_map:
        raise HTTPException(400, "Tipo de gráfico no reconocido")
    plot_type = plot_map[plot_type_raw]


    respuesta_sql = " ".join(lines[1:]).strip()
    sql_lower = respuesta_sql.lower()
    forbidden = ["insert", "update", "delete", "drop", "alter", "truncate", "create", "grant", "revoke"]


    if not sql_lower.startswith("select") or any(w in sql_lower for w in forbidden) or "limit" not in sql_lower:
        raise HTTPException(400, "Consulta SQL no permitida o falta LIMIT")
    with get_connection() as conn:
        df = pd.read_sql(respuesta_sql, conn)
    if df.empty:
        raise HTTPException(404, "No hay datos para generar el gráfico")


    fig, ax = plt.subplots(figsize=(7,5))
    if plot_type == "hist":
        ax.hist(df.iloc[:,0])
        ax.set_xlabel(df.columns[0])
        ax.set_title("Histograma")
    elif plot_type == "scatter":
        if df.shape[1] < 2:
            raise HTTPException(400, "Scatterplot requiere dos columnas")
        ax.scatter(df.iloc[:,0], df.iloc[:,1])
        ax.set_xlabel(df.columns[0])
        ax.set_ylabel(df.columns[1])
        ax.set_title("Scatterplot")
    elif plot_type == "box":
        ax.boxplot(df.iloc[:,0])
        ax.set_ylabel(df.columns[0])
        ax.set_title("Boxplot")


    buffer_bytes = BytesIO()
    fig.savefig(buffer_bytes, format="png")
    plt.close(fig)
    buffer_bytes.seek(0)
    return StreamingResponse(content=buffer_bytes, media_type="image/png")

